# Análise Gold — Motivos e Canais (V-Credit)

## Objetivo
Analisar os chamados por **motivo** e **canal**, usando as dimensões `vcredit.gold.dim_motivo` e `vcredit.gold.dim_canal`, e produzir indicadores que possam ser explorados no Power BI.

A análise cruza as dimensões com a tabela de chamados da Silver e agrega pesquisa de satisfação e custos por chamado **antes** dos JOINs. Isso evita multiplicar custos ou pesquisas quando há mais de um registro associado ao mesmo chamado.

In [0]:
from pyspark.sql import functions as F

catalog = "vcredit"
silver = f"{catalog}.silver"
gold = f"{catalog}.gold"

chamados_table = f"{silver}.tb_chamados"
satisfacao_table = f"{silver}.pesquisa_satisfacao"
custos_table = f"{silver}.custos"
dim_motivo_table = f"{gold}.dim_motivo"
dim_canal_table = f"{gold}.dim_canal"

print("Tabelas de entrada:")
for t in [chamados_table, satisfacao_table, custos_table, dim_motivo_table, dim_canal_table]:
    print("-", t)

## 1. Validação de tabelas e colunas

Antes da análise, confirmamos se as dimensões e tabelas de apoio estão disponíveis. Se o nome da tabela Silver de canais, motivos ou chamados variar no workspace, ajuste a variável correspondente acima.

In [0]:
required_columns = {
    chamados_table: [
        "id_chamado",
        "motivo",
        "canal",
        "id_atendente"
    ],
    satisfacao_table: [
        "id_chamado",
        "nota_atendimento"
    ],
    custos_table: [
        "id_chamado",
        "custo"
    ],
    dim_motivo_table: [
        "id_motivo",
        "nome_motivo",
        "categoria",
        "criticidade"
    ],
    dim_canal_table: [
        "id_canal",
        "nome_canal",
        "tipo_canal",
        "status_canal"
    ],
}

for table_name, expected in required_columns.items():
    try:
        df_check = spark.table(table_name)
    except Exception as e:
        raise ValueError(
            f"Não foi possível ler {table_name}. "
            "Confira se a tabela existe e se o nome está correto."
        ) from e

    missing = sorted(set(expected) - set(df_check.columns))

    if missing:
        raise ValueError(
            f"A tabela {table_name} não possui as colunas esperadas: "
            f"{missing}. Colunas encontradas: {df_check.columns}"
        )

print("Validação concluída: tabelas e colunas necessárias encontradas.")

## 2. Preparação dos dados

- Mantemos uma linha por chamado na base principal.
- Consolidamos pesquisas por chamado usando a média das notas e a quantidade de respostas.
- Normalizamos valores monetários para número, removendo `R$`, espaços e convertendo vírgula decimal.
- Consolidamos custos por chamado antes dos JOINs, evitando duplicação de valores.

In [0]:

df_chamados_raw = spark.table(chamados_table).alias("ch")
df_dim_motivo = spark.table(dim_motivo_table).alias("m")
df_dim_canal = spark.table(dim_canal_table).alias("c")

spark.table(chamados_table).printSchema()

df_chamados = (
    df_chamados_raw
    .join(
        df_dim_motivo,
        F.trim(F.col("ch.motivo")) == F.trim(F.col("m.nome_motivo")),
        "left"
    )
    .join(
        df_dim_canal,
        F.trim(F.col("ch.canal")) == F.trim(F.col("c.nome_canal")),
        "left"
    )
    .select(
        F.col("ch.id_chamado").cast("long").alias("id_chamado"),
        F.col("m.id_motivo").cast("int").alias("id_motivo"),
        F.col("m.nome_motivo").alias("nome_motivo"),
        F.col("m.categoria").alias("categoria"),
        F.col("m.criticidade").alias("criticidade"),
        F.col("c.id_canal").cast("int").alias("id_canal"),
        F.col("c.nome_canal").alias("nome_canal"),
        F.col("c.tipo_canal").alias("tipo_canal"),
        F.col("c.status_canal").alias("status_canal"),
        F.col("ch.id_atendente").cast("int").alias("id_atendente"),
        F.col("ch.motivo").alias("motivo_origem"),
        F.col("ch.canal").alias("canal_origem"),
        F.col("ch.resolvido").alias("resolvido"),
        F.col("ch.hora_abertura_chamado").alias("hora_abertura_chamado"),
        F.col("ch.hora_inicio_atendimento").alias("hora_inicio_atendimento"),
        F.col("ch.hora_finalizacao_atendimento").alias("hora_finalizacao_atendimento"),
        F.col("ch.tempo_espera").alias("tempo_espera"),
        F.col("ch.tempo_atendimento").alias("tempo_atendimento")
    )
    .dropDuplicates(["id_chamado"])
)

df_satisfacao = (
    spark.table(satisfacao_table)
    .select(
        F.col("id_chamado").cast("long").alias("id_chamado"),
        F.regexp_replace(
            F.col("nota_atendimento").cast("string"), ",", "."
        ).cast("double").alias("nota_atendimento")
    )
    .groupBy("id_chamado")
    .agg(
        F.avg("nota_atendimento").alias("nota_media"),
        F.count("nota_atendimento").alias("qtd_respostas_satisfacao"),
        F.sum(
            F.when(F.col("nota_atendimento").isNotNull(), 1).otherwise(0)
        ).alias("qtd_notas_preenchidas")
    )
)

df_custos_raw = spark.table(custos_table)

custo_txt = F.trim(F.col("custo").cast("string"))
custo_txt = F.regexp_replace(custo_txt, r"(?i)R\$", "")
custo_txt = F.regexp_replace(custo_txt, r"\s+", "")

custo_norm = F.when(
    custo_txt.contains(","),
    F.regexp_replace(
        F.regexp_replace(custo_txt, r"\.", ""),
        ",",
        "."
    )
).otherwise(custo_txt)

df_custos = (
    df_custos_raw
    .select(
        F.col("id_chamado").cast("long").alias("id_chamado"),
        custo_norm.cast("double").alias("custo_num")
    )
    .groupBy("id_chamado")
    .agg(
        F.sum("custo_num").alias("custo_total_chamado"),
        F.count("custo_num").alias("qtd_registros_custo")
    )
)

df_base = (
    df_chamados
    .join(df_satisfacao, on="id_chamado", how="left")
    .join(df_custos, on="id_chamado", how="left")
    .withColumn(
        "custo_total_chamado",
        F.coalesce(F.col("custo_total_chamado"), F.lit(0.0))
    )
)

print(f"Chamados distintos na base: {df_chamados.count():,}")
print(f"Chamados após JOINs: {df_base.select('id_chamado').distinct().count():,}")

display(df_base.limit(10))

## 3. Qualidade do relacionamento dimensional

Esta etapa mede quantos chamados não encontraram correspondência nas dimensões. Uma taxa alta pode indicar chaves ausentes, IDs inconsistentes ou dimensão desatualizada; esses registros devem ser investigados antes de publicar indicadores no Power BI.

In [0]:
df_integridade = df_base.agg(
    F.countDistinct("id_chamado").alias("qtd_chamados"),
    F.sum(
        F.when(F.col("id_motivo").isNull(), 1).otherwise(0)
    ).alias("chamados_sem_motivo_correspondente"),

    F.sum(
        F.when(F.col("id_canal").isNull(), 1).otherwise(0)
    ).alias("chamados_sem_canal_correspondente"),

    F.sum(
        F.when(F.col("nota_media").isNotNull(), 1).otherwise(0)
    ).alias("chamados_com_pesquisa"),

    F.sum(
        F.when(F.col("custo_total_chamado") > 0, 1).otherwise(0)
    ).alias("chamados_com_custo_positivo")
)

display(df_integridade)

## 4. Análise por motivo

Indicadores:
- quantidade e participação dos chamados;
- nota média e quantidade de pesquisas;
- custo total e custo médio por chamado;
- criticidade e categoria do motivo.

A participação percentual ajuda a identificar os motivos mais frequentes. A nota média deve ser interpretada junto da quantidade de respostas, pois motivos com poucas pesquisas podem apresentar médias instáveis.

In [0]:
total_chamados = df_base.select("id_chamado").distinct().count()

df_analise_motivo = (
    df_base
    .groupBy("id_motivo", "nome_motivo", "categoria", "criticidade")
    .agg(
        F.countDistinct("id_chamado").alias("qtd_chamados"),
        F.round(F.avg("nota_media"), 2).alias("nota_media_satisfacao"),
        F.sum(F.coalesce(F.col("qtd_respostas_satisfacao"), F.lit(0))).alias("qtd_respostas_satisfacao"),
        F.round(F.sum("custo_total_chamado"), 2).alias("custo_total"),
        F.round(F.avg("custo_total_chamado"), 2).alias("custo_medio_por_chamado"),
    )
    .withColumn("participacao_chamados_pct", F.round(F.col("qtd_chamados") * 100.0 / F.lit(total_chamados), 2))
    .orderBy(F.desc("qtd_chamados"))
)

display(df_analise_motivo)

## 5. Análise por canal

Comparamos canais individualmente e por tipo (autosserviço ou humano, conforme cadastro da dimensão). O objetivo é verificar volume, satisfação e custo médio; os números, por si só, não provam que um canal causa melhor ou pior atendimento.

In [0]:
df_analise_canal = (
    df_base
    .groupBy("id_canal", "nome_canal", "tipo_canal", "status_canal")
    .agg(
        F.countDistinct("id_chamado").alias("qtd_chamados"),
        F.round(F.avg("nota_media"), 2).alias("nota_media_satisfacao"),
        F.sum(F.coalesce(F.col("qtd_respostas_satisfacao"), F.lit(0))).alias("qtd_respostas_satisfacao"),
        F.round(F.sum("custo_total_chamado"), 2).alias("custo_total"),
        F.round(F.avg("custo_total_chamado"), 2).alias("custo_medio_por_chamado"),
    )
    .withColumn("participacao_chamados_pct", F.round(F.col("qtd_chamados") * 100.0 / F.lit(total_chamados), 2))
    .orderBy(F.desc("qtd_chamados"))
)

display(df_analise_canal)

## 6. Criticidade × satisfação e custo

Esta visão ajuda a priorizar motivos de criticidade alta com grande volume, nota média baixa ou custo médio elevado. Os limites de “nota baixa” e “custo alto” devem ser definidos com a área de negócio; por enquanto, a tabela ordena os casos para inspeção, sem criar um corte arbitrário.

In [0]:
df_priorizacao = (
    df_analise_motivo
    .orderBy(
        F.when(F.col("criticidade") == "Alta", 0)
         .when(F.col("criticidade") == "Média", 1)
         .otherwise(2),
        F.desc("qtd_chamados"),
        F.asc("nota_media_satisfacao"),
        F.desc("custo_medio_por_chamado"),
    )
)

display(df_priorizacao)

## 7. Cruzamento entre motivo e canal

Mostra quais motivos aparecem com maior frequência em cada canal. Essa matriz pode revelar, por exemplo, se certos tipos de solicitação estão concentrados no autosserviço ou no atendimento humano. A interpretação final deve considerar também satisfação, custo e criticidade.

In [0]:
df_motivo_canal = (
    df_base
    .groupBy(
        "categoria", "nome_motivo", "criticidade",
        "nome_canal", "tipo_canal", "status_canal"
    )
    .agg(
        F.countDistinct("id_chamado").alias("qtd_chamados"),
        F.round(F.avg("nota_media"), 2).alias("nota_media_satisfacao"),
        F.round(F.avg("custo_total_chamado"), 2).alias("custo_medio_por_chamado"),
    )
    .orderBy(F.desc("qtd_chamados"))
)

display(df_motivo_canal)